# 🚀 D4UC (Discord 4-Character Usernames Checker) — Google Colab Edition

> **Chrome 131 TLS 스텔스 위장 + Surfshark 1:1 다중 국가 터널 풀 + Google Drive 연동 + Discord 웹훅 주기 리포트**
>
> 내 PC나 스마트폰을 켜두지 않고, **구글 클라우드(Colab)의 무료 고속 네트워크**를 이용해 24시간 무중단으로 디스코드 레어 닉네임을 전수검사합니다.
>
> ✅ **Google Drive 연동**: 이전 세션 DB 자동 복원으로 중복 검사 0% 방지 + 발견된 닉네임(`available.txt`) 즉시 Drive 원자적 기록 + SQLite WAL 30초 주기 백업
> ✅ **Discord 웹훅 다중 알림**: 가용(된거) 닉네임 실시간 발견 핑(@here) + 5분 주기 진행상황/통계/사용중(안된거)/429 종합 리포트

---
### 💡 구글 코랩 사용 순서 (4단계 요약)
1. **[1단계]** 환경 설정 셀 실행 (`Shift + Enter` 또는 ▶ 버튼)
2. **[2단계]** 아래 대화형 UI(폼)에서 모드/글자수/워커수/웹훅/Google Drive 설정
3. **[3단계]** 검사 시작 셀 실행 -> 실시간 로그 스트리밍 및 디스코드 웹훅 수신 확인
4. **[4단계]** 발견된 닉네임 확인 및 Google Drive 영구 보존 / 로컬 다운로드


In [1]:
#@title 🚀 1단계: D4UC 설치 및 환경 준비
#@markdown 구글 코랩 환경에 D4UC 코어 엔진, Chrome131 TLS 모듈 및 Linux x86_64 Wireproxy를 원클릭 설치합니다.

import os
import sys

# 1. 저장소 클론 또는 최신 코드 확인
REPO_DIR = "/content/Discord-4Character-Usernames-Checker"
if not os.path.exists(REPO_DIR):
    print("[*] D4UC 저장소를 복제하는 중...")
    !git clone https://github.com/camt4372/Discord-4Character-Usernames-Checker.git $REPO_DIR
else:
    print("[*] 기존 저장소 발견, 최신 코드로 동기화합니다...")
    !cd $REPO_DIR && git pull origin main

%cd $REPO_DIR

# 2. 원클릭 코랩 셋업 스크립트 실행
!chmod +x colab_setup.sh
!./colab_setup.sh

print("\n✅ [완료] D4UC 구글 코랩 환경 준비가 100% 완료되었습니다!")

[*] D4UC 저장소를 복제하는 중...
Cloning into '/content/Discord-4Character-Usernames-Checker'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 17 (delta 2), reused 5 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 12.32 KiB | 2.46 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/Discord-4Character-Usernames-Checker
chmod: cannot access 'colab_setup.sh': No such file or directory
/bin/bash: line 1: ./colab_setup.sh: No such file or directory

✅ [완료] D4UC 구글 코랩 환경 준비가 100% 완료되었습니다!


In [ ]:
#@title ⚙️ 2단계: D4UC 검사 옵션 설정 (Interactive Form)
#@markdown 아래 옵션을 마우스로 선택하세요. (코드를 수정할 필요 없이 오른쪽 폼에서 바로 설정됩니다)

MODE = "hybrid" #@param ["hybrid", "permutation", "dictionary", "pattern", "numeric", "custom_mask", "combinatorial", "recheck"]
LENGTH = 4 #@param {type:"integer"}
WORKERS = 5 #@param {type:"slider", min:1, max:20, step:1}
SHARD = "None" #@param ["None", "2_1", "2_2", "3_1", "3_2", "3_3", "4_1", "4_2", "4_3", "4_4"]
PROXY_MODE = "surfshark" #@param ["surfshark", "direct"]
SURFSHARK_PRIVATE_KEYS = "" #@param {type:"string"}
DISCORD_WEBHOOK_URL = "https://discord.com/api/webhooks/1574452728447373493/PJbrH9HazAlaLUXRCb8piq_Oxi4mQa8FrprPYZqsizQPhmhvgaNIcUupYIYdHMxfIyAQ" #@param {type:"string"}
WEBHOOK_PROGRESS_REPORT = True #@param {type:"boolean"}
WEBHOOK_REPORT_INTERVAL_MIN = 5 #@param {type:"integer"}
SAVE_TO_GOOGLE_DRIVE = True #@param {type:"boolean"}
RESTORE_PREVIOUS_HISTORY = True #@param {type:"boolean"}

# Google Drive 영구 저장소 연동 및 이전 이력 복원 확인
DRIVE_DATA_DIR = "/content/drive/MyDrive/D4UC"
if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    print("[*] Google Drive를 마운트하여 결과를 영구 보존하고 중복 검사를 0%로 방지합니다...")
    drive.mount('/content/drive')
    os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
    print(f"  ✅ 구글 드라이브 저장 경로 준비 완료: {DRIVE_DATA_DIR}")
    if RESTORE_PREVIOUS_HISTORY:
        prev_db = os.path.join(DRIVE_DATA_DIR, "d4uc_results.db")
        prev_txt = os.path.join(DRIVE_DATA_DIR, "available.txt")
        if os.path.exists(prev_db) or os.path.exists(prev_txt):
            print("  🔄 이전 세션 검사 이력(DB/available.txt)이 감지되었습니다. 실행 시 자동 복원되어 중복 검사를 건너뜁니다.")

print("\n[설정 요약]")
print(f" - 모드: {MODE.upper()} | 길이: {LENGTH}자 | 워커: {WORKERS}개")
print(f" - 분할 샤딩: {SHARD} | 프록시: {PROXY_MODE}")
if SURFSHARK_PRIVATE_KEYS.strip():
    print(" - Surfshark 와이어가드 개인키: 등록됨 (안전 마스킹)")
else:
    print(" - Surfshark 와이어가드 개인키: 미지정 (로컬 credentials.json 또는 기본키 탐색)")
if DISCORD_WEBHOOK_URL.strip():
    print(f" - 디스코드 웹훅 알림: 활성화 ('된거' 즉시 알림 + '안된거'/진행상황 {WEBHOOK_REPORT_INTERVAL_MIN}분 주기)")
print(f" - 구글 드라이브 실시간 연동: {'활성 (/D4UC/)' if SAVE_TO_GOOGLE_DRIVE else '비활성'}")


In [ ]:
#@title ⚡ 3단계: 대화형 Colab Web GUI 및 실시간 검사 가동 (Interactive Dashboard)
#@markdown 아래 셀을 실행하면 브라우저 내에 **Colab 전용 인터랙티브 다크 대시보드(Web GUI)**가 즉시 렌더링되며 검사가 시작됩니다.<br>
#@markdown 🛑 **작업을 중단하려면 셀 왼쪽의 정지(■) 버튼을 누르세요.**

import subprocess
import shlex
import re
import json
import time
from src.ui.colab_gui import ColabDashboard

# 1. Colab 전용 반응형 대시보드 초기화 및 렌더링
dashboard = ColabDashboard(lang="ko", title="D4UC Google Colab Cloud Engine")
dashboard.render()

# 2. 실행 명령어 조립
cmd = [
    sys.executable, "run.py",
    "--no-dashboard",
    "--mode", MODE,
    "--length", str(LENGTH),
    "--workers", str(WORKERS),
    "--proxy", PROXY_MODE
]

if SHARD != "None":
    cmd.extend(["--shard", SHARD])

if SURFSHARK_PRIVATE_KEYS.strip():
    cmd.extend(["--keys", SURFSHARK_PRIVATE_KEYS.strip()])

if DISCORD_WEBHOOK_URL.strip():
    cmd.extend(["--webhook", DISCORD_WEBHOOK_URL.strip()])
    if WEBHOOK_PROGRESS_REPORT:
        cmd.append("--webhook-progress")
        interval_sec = max(60, int(WEBHOOK_REPORT_INTERVAL_MIN * 60))
        cmd.extend(["--webhook-interval", str(interval_sec)])

if SAVE_TO_GOOGLE_DRIVE:
    cmd.extend(["--drive-dir", DRIVE_DATA_DIR])

cmd_display = ' '.join(cmd[:8])
dashboard.append_log(f"[*] 실행 명령어: {cmd_display} ...")
dashboard.append_log("🚀 D4UC Google Colab 클라우드 검사를 시작합니다!")

# 3. 하트비트 기록 헬퍼 (GAS 하이브리드 워치독 연동)
def write_colab_heartbeat():
    if SAVE_TO_GOOGLE_DRIVE and os.path.exists(DRIVE_DATA_DIR):
        try:
            hb_path = os.path.join(DRIVE_DATA_DIR, "colab_heartbeat.json")
            with open(hb_path, "w", encoding="utf-8") as f:
                json.dump({"status": "RUNNING", "timestamp_ms": int(time.time() * 1000), "source": "Colab"}, f)
        except Exception:
            pass

write_colab_heartbeat()
last_hb_time = time.time()

# 4. 프로세스 실행 및 실시간 스트리밍 대시보드 브릿지
total_checked = 0
total_avail = 0
total_taken = 0
total_429 = 0
total_skip = 0
current_rps = 0.0

try:
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in iter(process.stdout.readline, ''):
        line_clean = line.strip()
        if not line_clean:
            continue
        dashboard.append_log(line_clean)

        # 가용 닉네임 파싱
        if "AVAILABLE" in line_clean:
            m = re.search(r'([a-zA-Z0-9_\.]{3,6})\s+is\s+AVAILABLE', line_clean)
            uname = m.group(1) if m else "found"
            total_avail += 1
            dashboard.add_username(uname, time.strftime("%H:%M:%S"), 85, f"{len(uname)}char")
        elif "TAKEN" in line_clean:
            total_taken += 1
        elif "429" in line_clean or "Rate limit" in line_clean:
            total_429 += 1

        total_checked = total_avail + total_taken
        if time.time() - last_hb_time > 15:
            write_colab_heartbeat()
            last_hb_time = time.time()
            dashboard.update_metrics(current_rps, total_checked, total_avail, total_taken, total_429, total_skip)

    process.stdout.close()
    process.wait()
except KeyboardInterrupt:
    dashboard.append_log("\n🛑 [중단] 사용자가 검사를 중지했습니다. 프로세스를 안전하게 회수합니다...")
    process.terminate()
    process.wait()
    dashboard.append_log("✅ 안전하게 종료되었습니다.")
finally:
    # Google Drive 결과 최종 이중 백업
    if SAVE_TO_GOOGLE_DRIVE and os.path.exists(DRIVE_DATA_DIR):
        import shutil
        try:
            if os.path.exists("available.txt"):
                shutil.copyfile("available.txt", os.path.join(DRIVE_DATA_DIR, "available.txt"))
            if os.path.exists("data/d4uc_results.db"):
                shutil.copyfile("data/d4uc_results.db", os.path.join(DRIVE_DATA_DIR, "d4uc_results.db"))
            dashboard.append_log(f"\n💾 [동기화 확인] 결과 파일이 Google Drive에 안전하게 최종 보존되었습니다: {DRIVE_DATA_DIR}")
        except Exception as e:
            dashboard.append_log(f"\n⚠️ [Drive 백업 경고] {e}")



In [2]:
#@title 💾 4단계: 발견된 닉네임 확인 및 다운로드
#@markdown 지금까지 발견된 사용 가능한 레어 닉네임 목록을 확인하고 로컬 PC로 다운로드합니다.

from google.colab import files

print("=" * 60)
print("🎉 [발견된 사용 가능 닉네임 목록 (available.txt)]")
print("=" * 60)

if os.path.exists("available.txt") and os.path.getsize("available.txt") > 0:
    with open("available.txt", "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]
        for i, name in enumerate(lines, 1):
            print(f"  [{i:02d}] {name}")
        print(f"\n총 {len(lines)}개의 희귀 닉네임이 발견되었습니다!")

    # 파일 다운로드
    files.download("available.txt")
else:
    print("ℹ️ 아직 발견된 닉네임이 없거나 검사가 진행 중입니다.")
    if os.path.exists("data/d4uc_results.db"):
        print(f"  (참고: SQLite 데이터베이스 파일 크기: {os.path.getsize('data/d4uc_results.db')} 바이트)")

🎉 [발견된 사용 가능 닉네임 목록 (available.txt)]
ℹ️ 아직 발견된 닉네임이 없거나 검사가 진행 중입니다.
